# 101 — Raw File Audit

This notebook checks local file-managed inputs against the sizes and SHA-256
digests recorded in `config/raw_data_registry.json`. It reports missing files
and identity mismatches. Resources governed by provider manifests are listed
with the provenance rules used for their payloads.

In [1]:
# =============================================================================
# Load the raw data registry
# =============================================================================

import pandas as pd

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths
from pancancer_epigenetics.utils.raw_data_registry import (
    enumerate_file_managed_files,
    enumerate_manifest_managed_resources,
    load_raw_data_registry,
)

registry = load_raw_data_registry()

In [2]:
# =============================================================================
# Define the raw-file audit scope
# =============================================================================

file_managed_files = enumerate_file_managed_files(registry, root=Paths.root)
manifest_managed_resources = enumerate_manifest_managed_resources(registry)

print(f"File-managed raw files scheduled for hashing: {len(file_managed_files):,}")

if len(file_managed_files) > 1000:
    raise RuntimeError(
        "Unexpectedly large file-managed audit scope. "
        "Inspect the registry classification before hashing."
    )

print("Manifest-managed resources excluded from individual hashing:")
for resource in manifest_managed_resources:
    resource_name = resource["dataset_id"]
    if resource["resource_id"]:
        resource_name += f".{resource['resource_id']}"
    print(f"  - {resource_name}: {resource['reason']}")

File-managed raw files scheduled for hashing: 45
Manifest-managed resources excluded from individual hashing:
  - tcga.methylation_array_sesame_beta_values: TCGA GDC methylation payload identity is governed by the frozen GDC manifest, sample sheet, metadata export, cohort freeze, and provider file checksums; individual payload SHA256 is not duplicated here.
  - tcga.rnaseq_star_counts: TCGA GDC RNA-seq STAR-Counts payload identity is governed by the frozen GDC manifest, sample sheet, metadata export, cohort freeze, and provider file checksums; individual payload SHA256 is not duplicated here.
  - tcga.somatic_mutation_wxs_masked_maf: TCGA GDC somatic mutation payload identity is governed by the frozen GDC source exports, the exact-sample operational manifest, provider file size and MD5 checksums, and notebook 108 download validation; individual payload SHA256 is not duplicated here.


In [3]:
# =============================================================================
# Verify raw file sizes and SHA-256 digests
# =============================================================================

records = []
missing_files = []
size_mismatches = []
sha256_mismatches = []

for file_record in file_managed_files:
    file_path = file_record["path"]
    metadata = file_record["metadata"]
    relative_path = file_record["relative_path"]

    record = {
        "relative_path": relative_path,
        "dataset": file_record["dataset_id"],
        "resource": file_record["resource_id"],
        "provenance_mode": file_record["provenance_mode"],
        "status": file_record["status"],
        "role": file_record["role"],
        "file_name": file_record["file_name"],
        "registered_size_bytes": metadata["size_bytes"],
        "registered_sha256": metadata["sha256"],
    }

    if not file_path.is_file():
        missing_files.append(relative_path)
        record.update(
            {
                "exists": False,
                "observed_size_bytes": None,
                "observed_sha256": None,
            }
        )
        records.append(record)
        continue

    observed_size = file_path.stat().st_size
    observed_sha256 = calculate_sha256(file_path)

    record.update(
        {
            "exists": True,
            "observed_size_bytes": observed_size,
            "observed_sha256": observed_sha256,
        }
    )

    if observed_size != metadata["size_bytes"]:
        size_mismatches.append(relative_path)

    if observed_sha256 != metadata["sha256"]:
        sha256_mismatches.append(relative_path)

    records.append(record)

raw_file_audit = (
    pd.DataFrame.from_records(records)
    .sort_values("relative_path")
    .reset_index(drop=True)
)

raw_file_audit

,relative_path,dataset,resource,provenance_mode,status,role,file_name,registered_size_bytes,registered_sha256,exists,observed_size_bytes,observed_sha256
0,data/raw/ctrp/CTRPv2.0_2015_ctd2_ExpandedDatas...,ctrp,NaN,file_managed,acquired_and_used,primary_pharmacogenomic_archive,CTRPv2.0_2015_ctd2_ExpandedDataset.zip,342737645,8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301...,True,342737645,8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301...
1,data/raw/ctrp/PSet_CTRPv2.rds,ctrp,NaN,file_managed,acquired_not_used,secondary_harmonized_pharmacoset,PSet_CTRPv2.rds,40707609,95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade2...,True,40707609,95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade2...
2,data/raw/depmap/CRISPRGeneEffect.csv,depmap,NaN,file_managed,acquired_and_used,crispr_gene_effect_matrix,CRISPRGeneEffect.csv,428678699,3d8f3ec6dbf2db7ff834b79b508622ec0b226f3518003f...,True,428678699,3d8f3ec6dbf2db7ff834b79b508622ec0b226f3518003f...
3,data/raw/depmap/D2_combined_CL_data.csv,depmap_rnai_demeter2,NaN,file_managed,supporting,rnai_cell_line_model_parameters,D2_combined_CL_data.csv,68023,d515cbc05de2a3dc6cbdf352d44a97bb7618fc1ffb63e4...,True,68023,d515cbc05de2a3dc6cbdf352d44a97bb7618fc1ffb63e4...
4,data/raw/depmap/D2_combined_gene_dep_scores.csv,depmap_rnai_demeter2,NaN,file_managed,acquired_and_used,rnai_dependency_matrix,D2_combined_gene_dep_scores.csv,160616970,6bfe7b24ef191debccd502b7dfe61c4b0763caba3e02c6...,True,160616970,6bfe7b24ef191debccd502b7dfe61c4b0763caba3e02c6...
5,data/raw/depmap/Model.csv,depmap,NaN,file_managed,acquired_and_used,model_metadata,Model.csv,645696,b7a0c1385e6cef30132b56aff61f1261d11e3f490490b3...,True,645696,b7a0c1385e6cef30132b56aff61f1261d11e3f490490b3...
6,data/raw/depmap/OmicsDefaultModelProfiles.csv,depmap,NaN,file_managed,acquired_not_used,default_model_profiles,OmicsDefaultModelProfiles.csv,90080,096a96c39d88374cb7c37058816e6cde29d6617b820b8a...,True,90080,096a96c39d88374cb7c37058816e6cde29d6617b820b8a...
7,data/raw/depmap/OmicsExpressionProteinCodingGe...,depmap,NaN,file_managed,acquired_and_used,transcriptomic_expression_matrix,OmicsExpressionProteinCodingGenesTPMLogp1.csv,506628654,2a71dc94110efcc0221eae821bb93a9f03b54bea16f005...,True,506628654,2a71dc94110efcc0221eae821bb93a9f03b54bea16f005...
8,data/raw/depmap/OmicsProfiles.csv,depmap,NaN,file_managed,acquired_and_used,omics_profile_metadata,OmicsProfiles.csv,254733,0bb19ee57935d4b213d378a4db9caacaa1db7318a63ae8...,True,254733,0bb19ee57935d4b213d378a4db9caacaa1db7318a63ae8...
9,data/raw/depmap/OmicsSomaticMutations.csv,depmap,NaN,file_managed,acquired_and_used,somatic_mutation_metadata,OmicsSomaticMutations.csv,338945382,92e86ab6a8457ef99908daf2a2c82d9529320e23cd6c6b...,True,338945382,92e86ab6a8457ef99908daf2a2c82d9529320e23cd6c6b...


In [4]:
# =============================================================================
# Summarize raw-file audit results
# =============================================================================

audit_summary = pd.Series(
    {
        "files_checked": len(records),
        "missing_files": len(missing_files),
        "size_mismatches": len(size_mismatches),
        "sha256_mismatches": len(sha256_mismatches),
    },
    name="count",
)

audit_summary.to_frame()

,count
files_checked,45
missing_files,0
size_mismatches,0
sha256_mismatches,0
